# Experimenting with SHAP for explainability for the horizon architecture

**The goal**: Assess how well shapley values can explain predictions 

### Structure
- First cell sets up the data injestion and feature engineering for the horizon models
- Second cell trains the horizon model, and validates the model with Time-Series CV
- Third cell creates an interactive dashboard for testing the shap, implementing rule based text otputs providing simple explanations driven by shap values

### Findings
- zero-masked rollin features with lag provide the best insights
- sudden increases in last 7 days can be picked up by the model
- item_id could be dropped to make a better generalizable model

## Set up and Feature Engineering 

In [1]:
import polars as pl
import lightgbm as lgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import warnings
warnings.filterwarnings('ignore')

# --- 1. CONFIGURATION & FINANCIAL ASSUMPTIONS ---
DATA_PATH = "../../backend/data/processed/m5_improved.parquet"
NUM_FOLDS = 3
HORIZONS = range(1, 29)
ALPHA = 0.95 # We are explicitly testing safety stock (worst-case spikes)

# MVP Unit Economics
HOLDING_COST = 0.10 
MARGIN_PCT = 0.40   

def pinball_loss(y_true, y_pred, alpha):
    """Calculates the Quantile (Pinball) Loss."""
    error = y_true - y_pred
    return np.mean(np.maximum(alpha * error, (alpha - 1) * error))

# --- 2. LIGHTWEIGHT DATA LOADING ---
print("Loading 5% Data Sample for Fast Evaluation...")
df_full = pl.read_parquet(DATA_PATH)
max_d = df_full["d"].max()

#  We need an extra 56 days of history so our 28-day lags 
# and 28-day rolling means have enough data to calculate without turning into nulls.
train_start_d = max_d - (28 * NUM_FOLDS) - 365 - 56 

np.random.seed(42)
sampled_items = np.random.choice(df_full.select("item_id").unique().to_series().to_list(), size=int(df_full.select("item_id").n_unique() * 0.05), replace=False)
df_base = df_full.filter((pl.col("d") >= train_start_d) & pl.col("item_id").is_in(sampled_items)).sort(["store_id", "item_id", "d"])

# --- FEATURE ENGINEERING ---
print("⚙️ Engineering Time-Series Features (Lags & Momentum)...")
df_base = df_base.with_columns([
    # 1. Price Momentum
    (pl.col("sell_price") / pl.col("sell_price").shift(7).over(["store_id", "item_id"])).alias("price_momentum_7d"),
    (pl.col("sell_price") / pl.col("sell_price").shift(28).over(["store_id", "item_id"])).alias("price_momentum_28d"),
    
    # 2. Advanced Lags (Eliminates Data Leakage)
    pl.col("sales").shift(28).over(["store_id", "item_id"]).alias("lag_28")
])

# To calculate "Days Since Last Sale", we need a row counter grouped by item/store
df_base = df_base.with_columns(
    pl.int_range(1, pl.len() + 1).over(["store_id", "item_id"]).alias("row_nr")
)

# 3. Apply the Interpretability Momentum Features ON the 28-day lag
df_base = df_base.with_columns([
    # Standard Rolling Means
    pl.col("lag_28").rolling_mean(window_size=7).over(["store_id", "item_id"]).alias("roll_mean_7_lag_28"),
    pl.col("lag_28").rolling_mean(window_size=28).over(["store_id", "item_id"]).alias("roll_mean_28_lag_28"),

    # NEW: Zero-Masked 28-Day Mean (ignores 0s, forward-fills last known demand during stock-outs)
    pl.when(pl.col("lag_28") > 0).then(pl.col("lag_28")).otherwise(None)
      .rolling_mean(window_size=28, min_periods=1)
      .forward_fill()
      .over(["store_id", "item_id"]).alias("masked_roll_mean_28_lag_28"),

    # NEW: Exponential Moving Average (EMA) for smooth momentum decay
    pl.col("lag_28").ewm_mean(alpha=0.1, ignore_nulls=True).over(["store_id", "item_id"]).alias("ema_lag_28"),

    # NEW: Days Since Last Sale logic (Step 1: Tag the row number of the last sale)
    pl.when(pl.col("lag_28") > 0).then(pl.col("row_nr")).otherwise(None)
      .forward_fill()
      .over(["store_id", "item_id"]).alias("last_sale_row")
])

# Finalize the "Days Since Last Sale" feature
df_base = df_base.with_columns([
    (pl.col("row_nr") - pl.col("last_sale_row")).fill_null(0).alias("days_since_last_sale_lag_28")
])

# Clean up temp columns and drop nulls created by the shift
df_base = df_base.drop(["row_nr", "last_sale_row"]).drop_nulls(subset=["roll_mean_28_lag_28", "price_momentum_28d"])


# --- 4. FEATURE DEFINITIONS ---
features = [
    'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 
    'wday', 'month', 'sell_price', 'price_norm',
    'snap_CA', 'snap_TX', 'snap_WI', 
    'price_momentum_7d', 'price_momentum_28d',
    'lag_28', 'roll_mean_7_lag_28', 'roll_mean_28_lag_28',
    # Newly added interpretability features:
    'masked_roll_mean_28_lag_28', 'ema_lag_28', 'days_since_last_sale_lag_28'
]

cat_features = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]

# Convert strings to Categoricals for fast RAM training
for col in cat_features:
    df_base = df_base.with_columns(pl.col(col).cast(pl.String).cast(pl.Categorical).to_physical().alias(col))

Loading 5% Data Sample for Fast Evaluation...
⚙️ Engineering Time-Series Features (Lags & Momentum)...


## Horizon Model Training and Cross Validation

In [2]:
import numpy as np

print("Starting Time-Series Cross-Validation...")
cv_results = []

for fold in range(NUM_FOLDS):
    split_point = max_d - (fold + 1) * 28
    print(f"🔄 FOLD {fold + 1}/{NUM_FOLDS} | Evaluating Days {split_point + 1} to {split_point + 28}")

    horizon_dfs = [df_base.with_columns([pl.col("sales").shift(-h).over(["store_id", "item_id"]).alias("target"), pl.lit(h).cast(pl.Int16).alias("horizon_day")]).drop_nulls(subset=["target"]) for h in HORIZONS]
    df_global = pl.concat(horizon_dfs)
    X_train_glob = df_global.filter(df_global["d"] <= split_point).select(features + ['horizon_day']).to_pandas()
    y_train_glob = df_global.filter(df_global["d"] <= split_point).select("target").to_series().to_numpy()

    # --- 2. TRAIN MODELS ---
    model_glob = lgb.train(
        {'objective': 'quantile', 'alpha': ALPHA, 'learning_rate': 0.1, 'num_leaves': 64, 'verbosity': -1}, 
        lgb.Dataset(X_train_glob, y_train_glob, categorical_feature=cat_features, free_raw_data=False), 
        num_boost_round=150
    )

    # --- 3. INFERENCE & FINANCIAL SCORING ---
    day_0_data = df_base.filter(pl.col("d") == split_point)
    sell_prices = day_0_data.select("sell_price").to_series().to_numpy()
    
    fold_losses = [] # Track the quantile loss for each horizon in this fold

    for h in HORIZONS:
        actuals = df_global.filter((pl.col("d") == split_point) & (pl.col("horizon_day") == h)).select("target").to_series().to_numpy()
        preds_glob = model_glob.predict(day_0_data.with_columns(pl.lit(h).alias("horizon_day")).select(features + ['horizon_day']).to_pandas())
        
        # Calculate Quantile (Pinball) Loss for this specific horizon
        errors = actuals - preds_glob
        q_loss = np.where(errors >= 0, ALPHA * errors, (ALPHA - 1) * errors)
        mean_q_loss = np.mean(q_loss)
        
        fold_losses.append(mean_q_loss)

    # Calculate and print the overall average loss for the fold
    fold_avg_loss = np.mean(fold_losses)
    print(f"FOLD {fold + 1} COMPLETE | Average Quantile Loss: {fold_avg_loss:.4f}")
    
    # Store the results
    cv_results.append({
        'fold': fold + 1, 
        'loss': fold_avg_loss
    })

# Print the final overall performance across all folds
overall_mean_loss = np.mean([res['loss'] for res in cv_results])
print(f"CV COMPLETE | Overall Mean Quantile Loss: {overall_mean_loss:.4f}")

Starting Time-Series Cross-Validation...
🔄 FOLD 1/3 | Evaluating Days 1914 to 1941
FOLD 1 COMPLETE | Average Quantile Loss: 0.1843
🔄 FOLD 2/3 | Evaluating Days 1886 to 1913
FOLD 2 COMPLETE | Average Quantile Loss: 0.1852
🔄 FOLD 3/3 | Evaluating Days 1858 to 1885
FOLD 3 COMPLETE | Average Quantile Loss: 0.1946
CV COMPLETE | Overall Mean Quantile Loss: 0.1881


## Explainability Dashboard

- adjustable the input of several input values to see how the model handles features
- displays the actual sales history, and shap graph to visualize importance and how the model behaves

In [ ]:
# ==========================================
# CELL 6: "WHAT-IF" SCENARIO EXPLAINER DASHBOARD (WITH SALES HISTORY)
# ==========================================
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import shap

print("🎛️ Loading Interactive 'What-If' SHAP Dashboard...")

X_explain = X_train_glob.sample(n=min(5000, len(X_train_glob)), random_state=42).reset_index(drop=True)

print("Calculating SHAP values (this may take a moment)...")
explainer = shap.TreeExplainer(model_glob)
shap_values = explainer(X_explain)

# 1. Core Selection Controls
item_dropdown = widgets.Dropdown(options=sorted(X_explain['item_id'].unique()), description='Item ID:')
store_dropdown = widgets.Dropdown(options=sorted(X_explain['store_id'].unique()), description='Store ID:')
load_button = widgets.Button(description='1. Load Current Data', button_style='primary')

# 2. Containers for Dynamic UI
controls_box = widgets.VBox() 
analyze_button = widgets.Button(description='2. Run What-If Scenario', button_style='success')
analyze_button.layout.display = 'none' 
output = widgets.Output()

state = {'base_row': None, 'input_widgets': {}}
tweakable_cols = [c for c in ['sell_price','masked_roll_mean_28_lag_28' ,'roll_mean_28_lag_28', 'roll_mean_7_lag_28', 'price_norm'] if c in X_explain.columns]

# --- NEW HELPER FUNCTION: Plot Historical Sales ---
def plot_actual_sales(item, store):
    try:
        # Access df_base from the global environment to get actual target sales
        # We grab the last 90 days of history for visual clarity
        history = df_base.filter(
            (pl.col("item_id") == item) & (pl.col("store_id") == store)
        ).select(["d", "sales"]).tail(90).to_pandas()

        if not history.empty:
            plt.figure(figsize=(10, 3))
            plt.bar(history['d'], history['sales'], color='#2ca02c', alpha=0.7, edgecolor='black')
            plt.axhline(history['sales'].mean(), color='red', linestyle='--', label=f"Mean: {history['sales'].mean():.1f}")
            plt.title(f"Actual Sales History (Last 90 Days) - Item {item} at Store {store}", fontsize=12, fontweight='bold')
            plt.xlabel("Day (d)")
            plt.ylabel("Units Sold")
            plt.legend()
            plt.tight_layout()
            plt.show()
        else:
            print("⚠️ No historical sales data found in df_base for this combination.")
    except Exception as e:
        print(f"⚠️ Could not load historical sales graph. Ensure 'df_base' is in memory. Error: {e}")
# ------------------------------------------------

def on_load_clicked(b):
    with output:
        clear_output(wait=True)
        TARGET_ITEM = item_dropdown.value
        TARGET_STORE = store_dropdown.value

        matching_rows = X_explain[(X_explain['item_id'] == TARGET_ITEM) & (X_explain['store_id'] == TARGET_STORE)]

        if matching_rows.empty:
            print("⚠️ This specific item/store combination is not in your current data sample. Please try another.")
            analyze_button.layout.display = 'none'
            controls_box.children = []
            return

        state['base_row'] = matching_rows.iloc[[0]].copy()

        print(f"✅ Real data loaded for Item {TARGET_ITEM} at Store {TARGET_STORE}.")
        
        # Draw the historical sales chart immediately so the user has context
        plot_actual_sales(TARGET_ITEM, TARGET_STORE)
        
        print("🛠️ Change the values below to simulate a new scenario:")

        widget_list = []
        state['input_widgets'] = {}
        for col in tweakable_cols:
            actual_val = state['base_row'][col].iloc[0]
            w = widgets.FloatText(value=actual_val, description=col + ':', style={'description_width': 'initial'})
            state['input_widgets'][col] = w
            widget_list.append(w)

        controls_box.children = widget_list
        analyze_button.layout.display = 'block' 

def on_analyze_clicked(b):
    with output:
        clear_output(wait=True)
        TARGET_ITEM = item_dropdown.value
        TARGET_STORE = store_dropdown.value
        
        print("⚙️ Running Simulated SHAP Analysis...")

        # Redraw the history plot so it stays visible above the SHAP waterfall
        plot_actual_sales(TARGET_ITEM, TARGET_STORE)

        modified_row = state['base_row'].copy()
        for col, w in state['input_widgets'].items():
            modified_row[col] = w.value

        single_shap_value = explainer(modified_row)

        plt.figure(figsize=(10, 6))
        plt.title(f"Simulated SHAP Waterfall: Item {TARGET_ITEM} | Store {TARGET_STORE}", fontsize=14, fontweight='bold')
        shap.plots.waterfall(single_shap_value[0], max_display=10, show=False)
        plt.tight_layout()
        plt.show()

        base_value = single_shap_value[0].base_values
        final_prediction = single_shap_value[0].values.sum() + base_value

        impact_df = pd.DataFrame({
            'Feature': modified_row.columns,
            'Simulated_Value': single_shap_value[0].data,
            'SHAP_Impact': single_shap_value[0].values
        })

        impact_df['Abs_Impact'] = impact_df['SHAP_Impact'].abs()
        impact_df = impact_df.sort_values(by='Abs_Impact', ascending=False).drop(columns=['Abs_Impact'])

        top_feature = impact_df.iloc[0]
        direction = "increased" if top_feature['SHAP_Impact'] > 0 else "decreased"

        print("\n--- 📝 SIMULATED SCENARIO EXPLANATION ---")
        print(f"The baseline 95th quantile forecast for an average item is {base_value:.2f} units.")
        print(f"Under these simulated conditions, the new safety stock forecast is {final_prediction:.2f} units.")
        print(f"\nThe dominant factor driving this simulated outcome is '{top_feature['Feature']}'.")
        
        actual_val = top_feature['Simulated_Value']
        val_display = f"{actual_val:.4f}" if isinstance(actual_val, float) else f"{actual_val}"
        
        print(f"By setting its value to {val_display}, it {direction} the forecast limit by {abs(top_feature['SHAP_Impact']):.2f} units.")

load_button.on_click(on_load_clicked)
analyze_button.on_click(on_analyze_clicked)

display(widgets.HBox([item_dropdown, store_dropdown, load_button]))
display(controls_box)
display(analyze_button)
display(output)

🎛️ Loading Interactive 'What-If' SHAP Dashboard...
Calculating SHAP values (this may take a moment)...


VBox()

Button(button_style='success', description='2. Run What-If Scenario', layout=Layout(display='none'), style=But…

Output()